# LangChain y LangSmith · RAG desde cero

La generación aumentada por recuperación (RAG) conecta un modelo de lenguaje con documentos propios: se **indexan** los documentos (trocear, calcular *embeddings* y guardarlos en un almacén vectorial), se **recuperan** los fragmentos más parecidos a la pregunta y el modelo **genera** la respuesta con ese contexto. Después se mejora la recuperación transformando la pregunta: *multi-query*, RAG-Fusion, descomposición, *step-back* y HyDE.

El recorrido sigue las partes 1 a 9 de [RAG From Scratch](https://github.com/langchain-ai/rag-from-scratch) (LangChain), adaptado para ejecutarse sin claves de API: los *embeddings* se calculan con un `HashingVectorizer` de `scikit-learn` sobre raíces de palabras y el modelo de chat es local y determinista. Las tareas en las que un LLM real escribiría texto nuevo (reformular preguntas, escribir un documento hipotético) se simulan con reglas y unas pocas frases preparadas. El corpus son doce artículos breves sobre agentes, RAG y evaluación.

In [1]:
ARTICULOS = [
    ("Agentes basados en LLM", "agentes", 2023, "Ana Pérez", 1200,
     "Un agente autónomo usa un modelo de lenguaje como cerebro. El agente combina planificación, memoria y uso de herramientas para resolver tareas largas. El modelo decide en cada paso qué acción ejecutar y observa el resultado antes de continuar."),
    ("Planificación y descomposición de tareas", "agentes", 2023, "Ana Pérez", 950,
     "La planificación divide una tarea compleja en subtareas más pequeñas y manejables. La cadena de pensamiento pide al modelo que razone paso a paso antes de responder. El árbol de pensamientos explora varias alternativas de razonamiento en cada paso y evalúa cuál conviene seguir."),
    ("Reflexión y autocrítica", "agentes", 2024, "Luis Santana", 610,
     "La reflexión permite a un agente corregir sus errores a partir de la experiencia. ReAct intercala pensamientos, acciones y observaciones en un mismo bucle. Reflexion guarda en memoria un resumen de los fallos anteriores para no repetirlos en el siguiente intento."),
    ("Memoria de los agentes", "agentes", 2024, "Luis Santana", 780,
     "La memoria a corto plazo es el contexto de la conversación que cabe en la ventana del modelo. La memoria a largo plazo guarda información en un almacén vectorial externo y la recupera por similitud. Los algoritmos de vecinos más cercanos aproximados, como HNSW o FAISS, aceleran esa búsqueda en millones de vectores."),
    ("Uso de herramientas", "agentes", 2025, "Carmen Díaz", 1500,
     "Las herramientas amplían lo que un modelo puede hacer: calculadoras, buscadores, intérpretes de código o APIs. El modelo genera una llamada a herramienta con argumentos estructurados y el sistema la ejecuta. Toolformer y MRKL son dos propuestas tempranas que enseñaron a los modelos a usar herramientas."),
    ("Qué es RAG", "rag", 2024, "Carmen Díaz", 2100,
     "La generación aumentada por recuperación, o RAG, conecta un modelo de lenguaje con documentos externos. Primero se recuperan los fragmentos relevantes para la pregunta y después el modelo genera la respuesta usando ese contexto. RAG reduce las alucinaciones y permite usar información privada o reciente sin reentrenar el modelo."),
    ("Indexación y troceado", "rag", 2024, "Pedro Rodríguez", 870,
     "Para indexar, los documentos se dividen en fragmentos de unos cientos de caracteres con cierto solapamiento. Cada fragmento se convierte en un vector numérico mediante un modelo de embeddings. Los vectores se guardan en un almacén vectorial junto con el texto y sus metadatos."),
    ("Recuperación por similitud", "rag", 2025, "Pedro Rodríguez", 640,
     "La recuperación compara el vector de la pregunta con los vectores de los fragmentos. La similitud del coseno mide el ángulo entre dos vectores y es la métrica más habitual. La búsqueda por máxima relevancia marginal, MMR, equilibra la relevancia con la diversidad de los resultados."),
    ("Transformación de consultas", "rag", 2025, "Ana Pérez", 520,
     "Una pregunta mal formulada recupera documentos poco útiles. Multi-query genera varias versiones de la pregunta y une los resultados. RAG-Fusion ordena los resultados combinados con la fusión de rangos recíprocos. HyDE genera un documento hipotético que responde a la pregunta y busca con su vector."),
    ("Reordenación y RAG correctivo", "rag", 2025, "Luis Santana", 410,
     "Un reordenador vuelve a puntuar los fragmentos recuperados con un modelo más preciso y se queda con los mejores. El RAG correctivo, CRAG, evalúa si los documentos recuperados son relevantes. Si no lo son, reescribe la consulta y busca en la web antes de generar la respuesta."),
    ("Evaluación de aplicaciones LLM", "evaluacion", 2025, "Carmen Díaz", 990,
     "Evaluar una aplicación con modelos de lenguaje exige un conjunto de ejemplos con entradas y salidas de referencia. Los evaluadores pueden ser reglas sencillas, métricas como la exactitud o un modelo que actúa como juez. LangSmith guarda los experimentos para comparar versiones de la aplicación."),
    ("Trazas y observabilidad", "evaluacion", 2026, "Pedro Rodríguez", 330,
     "Una traza registra cada paso de una aplicación: las llamadas al modelo, las herramientas y los recuperadores. Cada paso guarda sus entradas, salidas, duración y errores. Con las trazas se depuran fallos y se detectan los pasos más lentos o costosos."),
]

## 1. Documentos

Convertir cada artículo en un `Document` con el texto en `page_content` y el resto de campos (identificador, título, tema, año, autor y visitas) en `metadata`. Mostrar cuántos documentos y caracteres hay en total y cuántos documentos hay de cada tema.

In [2]:
from collections import Counter

from langchain_core.documents import Document

documentos = [
    Document(page_content=texto, metadata={"id": f"d{i:02d}", "titulo": titulo, "tema": tema, "anio": anio, "autor": autor, "vistas": vistas})
    for i, (titulo, tema, anio, autor, vistas, texto) in enumerate(ARTICULOS, 1)
]
print(len(documentos), "documentos,", sum(len(d.page_content) for d in documentos), "caracteres")
print(Counter(d.metadata["tema"] for d in documentos))
print(documentos[5])

12 documentos, 3407 caracteres
Counter({'agentes': 5, 'rag': 5, 'evaluacion': 2})
page_content='La generación aumentada por recuperación, o RAG, conecta un modelo de lenguaje con documentos externos. Primero se recuperan los fragmentos relevantes para la pregunta y después el modelo genera la respuesta usando ese contexto. RAG reduce las alucinaciones y permite usar información privada o reciente sin reentrenar el modelo.' metadata={'id': 'd06', 'titulo': 'Qué es RAG', 'tema': 'rag', 'anio': 2024, 'autor': 'Carmen Díaz', 'vistas': 2100}


## 2. Troceado

Dividir los documentos con `RecursiveCharacterTextSplitter` en fragmentos de 200 caracteres como máximo y 40 de solapamiento, cortando preferentemente entre frases (`separators` con `". "` y `keep_separator="end"`) y guardando la posición de inicio (`add_start_index=True`). Comprobar que ningún fragmento supera el tamaño y que cada uno coincide con el texto original en su posición. Mostrar los fragmentos de un documento y compararlos con los que salen cortando solo por espacios.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

troceador = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=40, separators=["\n\n", ". ", " ", ""], keep_separator="end", add_start_index=True
)
trozos = troceador.split_documents(documentos)
print(len(trozos), "fragmentos; tamaño medio:", round(sum(len(t.page_content) for t in trozos) / len(trozos)))

originales = {d.metadata["id"]: d.page_content for d in documentos}
for t in trozos:
    inicio = t.metadata["start_index"]
    assert len(t.page_content) <= 200
    assert originales[t.metadata["id"]][inicio:inicio + len(t.page_content)] == t.page_content

for t in (t for t in trozos if t.metadata["id"] == "d06"):
    print(f"[{t.metadata['start_index']:>3}] {t.page_content}")
print("Solo por espacios (con solapamiento):")
for t in RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40, separators=[" "]).split_documents(documentos[5:6]):
    print("     ", t.page_content)

25 fragmentos; tamaño medio: 136
[  0] La generación aumentada por recuperación, o RAG, conecta un modelo de lenguaje con documentos externos.
[104] Primero se recuperan los fragmentos relevantes para la pregunta y después el modelo genera la respuesta usando ese contexto.
[229] RAG reduce las alucinaciones y permite usar información privada o reciente sin reentrenar el modelo.
Solo por espacios (con solapamiento):
      La generación aumentada por recuperación, o RAG, conecta un modelo de lenguaje con documentos externos. Primero se recuperan los fragmentos relevantes para la pregunta y después el modelo genera la
      pregunta y después el modelo genera la respuesta usando ese contexto. RAG reduce las alucinaciones y permite usar información privada o reciente sin reentrenar el modelo.


## 3. Embeddings propios

Escribir una función `tokens(texto)` que pase a minúsculas, quite tildes y palabras vacías y recorte cada palabra a sus 5 primeras letras (una raíz aproximada: *agente* y *agentes* dan lo mismo). Con ella, crear la clase `EmbeddingsLocales`, que implementa la interfaz `Embeddings` de LangChain (`embed_documents` y `embed_query`) usando un `HashingVectorizer` normalizado. Calcular la matriz de similitud del coseno entre cuatro frases y comprobar que las dos que hablan de lo mismo con otras palabras son las más parecidas.

In [4]:
import re
import unicodedata

import numpy as np
from langchain_core.embeddings import Embeddings
from sklearn.feature_extraction.text import HashingVectorizer

VACIAS = set("""a al algo ante antes como con cual cuales de del desde donde el ella ellos en entre es esa ese esta este esto
hay la las le les lo los mas me mi muy no o otra otro para pero poco por que se sin sobre son su sus tambien un una uno unos
unas y ya ha han hace cada cierto ese eso esos sea ser si""".split())


def sin_tildes(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")


def tokens(texto):
    palabras = re.findall(r"\w+", sin_tildes(texto.lower()))
    return [p[:5] for p in palabras if p not in VACIAS and len(p) > 1]


class EmbeddingsLocales(Embeddings):
    def __init__(self, dimension=2**12):
        self.vectorizador = HashingVectorizer(analyzer=tokens, n_features=dimension, alternate_sign=False, norm="l2")

    def embed_documents(self, textos):
        return self.vectorizador.transform(textos).toarray().tolist()

    def embed_query(self, texto):
        return self.embed_documents([texto])[0]


embeddings = EmbeddingsLocales()
print(tokens("Los agentes recuperan información de la memoria a largo plazo."))
frases = [
    "El agente guarda sus recuerdos en la memoria.",
    "La memoria del agente guarda los recuerdos de las conversaciones.",
    "Se divide el documento en fragmentos con solapamiento.",
    "El reordenador puntúa de nuevo los fragmentos.",
]
matriz = np.array(embeddings.embed_documents(frases))
similitud = matriz @ matriz.T
print(np.round(similitud, 2))
np.fill_diagonal(similitud, 0)
assert np.unravel_index(similitud.argmax(), similitud.shape) in [(0, 1), (1, 0)]

['agent', 'recup', 'infor', 'memor', 'largo', 'plazo']
[[1.   0.89 0.   0.  ]
 [0.89 1.   0.   0.  ]
 [0.   0.   1.   0.25]
 [0.   0.   0.25 1.  ]]


## 4. Almacén vectorial

Indexar los fragmentos en un `InMemoryVectorStore` con los *embeddings* propios. Buscar los 3 fragmentos más parecidos a una pregunta con `similarity_search_with_score` y comprobar que el resultado coincide con una búsqueda por fuerza bruta (similitud del coseno con `numpy` sobre todos los fragmentos).

In [5]:
from langchain_core.vectorstores import InMemoryVectorStore

almacen = InMemoryVectorStore(embeddings)
ids = almacen.add_documents(trozos)
print(len(ids), "fragmentos indexados")

pregunta = "¿Dónde guarda un agente su memoria a largo plazo?"
resultados = almacen.similarity_search_with_score(pregunta, k=3)
for doc, puntuacion in resultados:
    print(f"{puntuacion:.3f} [{doc.metadata['titulo']}] {doc.page_content[:80]}...")

vectores = np.array(embeddings.embed_documents([t.page_content for t in trozos]))
fuerza_bruta = vectores @ np.array(embeddings.embed_query(pregunta))
mejores = np.argsort(-fuerza_bruta)[:3]
assert [trozos[i].page_content for i in mejores] == [d.page_content for d, _ in resultados]
assert np.allclose(fuerza_bruta[mejores], [p for _, p in resultados])

25 fragmentos indexados
0.572 [Memoria de los agentes] La memoria a corto plazo es el contexto de la conversación que cabe en la ventan...
0.325 [Agentes basados en LLM] Un agente autónomo usa un modelo de lenguaje como cerebro. El agente combina pla...
0.298 [Reflexión y autocrítica] Reflexion guarda en memoria un resumen de los fallos anteriores para no repetirl...


## 5. Recuperadores

Crear dos recuperadores (`as_retriever`) que devuelvan 4 fragmentos: uno por similitud y otro con máxima relevancia marginal (MMR, `fetch_k=12`, `lambda_mult=0.3`). Comparar para la misma pregunta los títulos que devuelve cada uno y cuántos documentos distintos cubren.

In [6]:
recuperador = almacen.as_retriever(search_kwargs={"k": 4})
recuperador_mmr = almacen.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 12, "lambda_mult": 0.3})

pregunta = "¿Qué es RAG?"
distintos = {}
for nombre, r in [("similitud", recuperador), ("MMR", recuperador_mmr)]:
    titulos = [d.metadata["titulo"] for d in r.invoke(pregunta)]
    distintos[nombre] = len(set(titulos))
    print(f"{nombre:>9}: {distintos[nombre]} documentos distintos -> {titulos}")
assert distintos["MMR"] > distintos["similitud"]

similitud: 3 documentos distintos -> ['Qué es RAG', 'Qué es RAG', 'Transformación de consultas', 'Reordenación y RAG correctivo']
      MMR: 4 documentos distintos -> ['Qué es RAG', 'Trazas y observabilidad', 'Recuperación por similitud', 'Indexación y troceado']


## 6. Generación

Crear `ModeloLocal`, un `BaseChatModel` que aplica reglas `(patrón, función)` al último mensaje y, si ninguna encaja, responde de forma extractiva: busca en el prompt las secciones `Contexto:` y `Pregunta:` y devuelve la frase del contexto que comparte más raíces con la pregunta (o "No lo sé" si no comparte ninguna).

Con él, montar la cadena RAG completa: `{"contexto": recuperador | formatear, "pregunta": RunnablePassthrough()} | prompt | modelo | StrOutputParser()`. Hacer una segunda versión que devuelva también las fuentes (títulos de los documentos usados) y responder tres preguntas.

In [7]:
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough


def frases_de(texto):
    return [f.strip() for f in re.split(r"(?<=[.!?])\s+", texto) if f.strip()]


def responder_con_contexto(prompt):
    partes = re.search(r"Contexto:\s*(.*?)\s*Pregunta:\s*(.*)", prompt, re.DOTALL)
    if not partes:
        return "No lo sé."
    clave = set(tokens(partes.group(2)))
    puntuadas = [(len(clave & set(tokens(f))), -i, f) for i, f in enumerate(frases_de(partes.group(1)))]
    mejor = max(puntuadas, default=(0, 0, ""))
    return mejor[2] if mejor[0] else "No lo sé."


class ModeloLocal(BaseChatModel):
    reglas: list = []

    @property
    def _llm_type(self) -> str:
        return "local"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        texto = messages[-1].text
        respuesta = next(
            (funcion(m) for patron, funcion in self.reglas if (m := re.search(patron, texto, re.IGNORECASE | re.DOTALL))),
            None,
        )
        respuesta = respuesta if respuesta is not None else responder_con_contexto(texto)
        return ChatResult(generations=[ChatGeneration(message=AIMessage(respuesta))])


def formatear(docs):
    return "\n\n".join(d.page_content for d in docs)


prompt_rag = ChatPromptTemplate.from_template(
    "Responde a la pregunta usando solo el contexto.\n\nContexto:\n{contexto}\n\nPregunta: {pregunta}"
)
modelo = ModeloLocal()
rag = {"contexto": recuperador | formatear, "pregunta": RunnablePassthrough()} | prompt_rag | modelo | StrOutputParser()

rag_con_fuentes = RunnableParallel(docs=recuperador, pregunta=RunnablePassthrough()).assign(
    respuesta=RunnableLambda(lambda x: {"contexto": formatear(x["docs"]), "pregunta": x["pregunta"]}) | prompt_rag | modelo | StrOutputParser()
)

for pregunta in ["¿Qué reduce RAG?", "¿Qué mide la similitud del coseno?", "¿Quién ganó el mundial de 2010?"]:
    salida = rag_con_fuentes.invoke(pregunta)
    print(f"{pregunta}\n  -> {salida['respuesta']}\n  fuentes: {sorted({d.metadata['titulo'] for d in salida['docs']})}")
    assert salida["respuesta"] == rag.invoke(pregunta)

¿Qué reduce RAG?
  -> RAG reduce las alucinaciones y permite usar información privada o reciente sin reentrenar el modelo.
  fuentes: ['Qué es RAG', 'Reordenación y RAG correctivo', 'Transformación de consultas']
¿Qué mide la similitud del coseno?
  -> La similitud del coseno mide el ángulo entre dos vectores y es la métrica más habitual.
  fuentes: ['Evaluación de aplicaciones LLM', 'Memoria de los agentes', 'Recuperación por similitud', 'Trazas y observabilidad']


¿Quién ganó el mundial de 2010?
  -> No lo sé.
  fuentes: ['Evaluación de aplicaciones LLM', 'Trazas y observabilidad']


## 7. Multi-query

Una misma idea se puede preguntar con palabras que no aparecen en los documentos. Pedir al modelo (una regla que sustituye sinónimos y extrae palabras clave) tres versiones de la pregunta, recuperar con cada una aplicando `recuperador.map()` y unir los resultados sin duplicados. Comparar los títulos recuperados con los de la pregunta original.

In [8]:
SINONIMOS = {
    "recuerda": "guarda en memoria", "errores": "fallos", "agente": "sistema autónomo", "herramientas": "APIs",
    "busca": "recupera", "trocea": "divide en fragmentos", "puntúa": "ordena", "textos": "documentos",
}


def versiones(m):
    pregunta = m.group(1).strip()
    sinonimos = pregunta
    for palabra, alternativa in SINONIMOS.items():
        sinonimos = re.sub(rf"\b{palabra}\b", alternativa, sinonimos)
    claves = " ".join(p for p in re.findall(r"\w+", pregunta) if sin_tildes(p.lower()) not in VACIAS)
    return "\n".join([sinonimos, f"Palabras clave: {claves}", f"¿Qué se sabe sobre {claves.lower()}?"])


modelo = ModeloLocal(reglas=[(r"Genera tres versiones de esta pregunta[^:]*:\s*(.*)", versiones)])
prompt_versiones = ChatPromptTemplate.from_template("Genera tres versiones de esta pregunta, una por línea: {pregunta}")
generar_versiones = prompt_versiones | modelo | StrOutputParser() | (lambda texto: texto.splitlines())


def union_unica(listas):
    vistos = {}
    for docs in listas:
        for d in docs:
            vistos.setdefault(d.page_content, d)
    return list(vistos.values())


multi_query = generar_versiones | recuperador.map() | union_unica

pregunta = "¿Cómo recuerda un agente sus errores?"
print(generar_versiones.invoke({"pregunta": pregunta}))
directos = {d.metadata["titulo"] for d in recuperador.invoke(pregunta)}
multiples = multi_query.invoke({"pregunta": pregunta})
print("Directa:", sorted(directos))
print("Multi-query:", len(multiples), "fragmentos ->", sorted({d.metadata["titulo"] for d in multiples}))
assert "Reflexión y autocrítica" in {d.metadata["titulo"] for d in multiples}

['¿Cómo guarda en memoria un sistema autónomo sus fallos?', 'Palabras clave: recuerda agente errores', '¿Qué se sabe sobre recuerda agente errores?']
Directa: ['Agentes basados en LLM', 'Reflexión y autocrítica', 'Trazas y observabilidad']
Multi-query: 7 fragmentos -> ['Agentes basados en LLM', 'Evaluación de aplicaciones LLM', 'Memoria de los agentes', 'Reflexión y autocrítica', 'Trazas y observabilidad']


## 8. RAG-Fusion

En lugar de unir sin más, ordenar los fragmentos de todas las versiones con la fusión de rangos recíprocos (RRF): cada fragmento suma $\frac{1}{k + r}$ por cada lista en la que aparece en la posición $r$ (con $k = 60$). Implementar `rrf` y mostrar los 4 mejores con su puntuación. Comprobar con un ejemplo pequeño que un documento que aparece en varias listas supera a uno que solo es primero en una.

In [9]:
def rrf(listas, k=60):
    puntuaciones, docs = {}, {}
    for lista in listas:
        for rango, d in enumerate(lista, 1):
            puntuaciones[d.page_content] = puntuaciones.get(d.page_content, 0) + 1 / (k + rango)
            docs[d.page_content] = d
    return [(docs[t], p) for t, p in sorted(puntuaciones.items(), key=lambda x: -x[1])]


a, b, c = (Document(page_content=x) for x in "abc")
assert [d.page_content for d, _ in rrf([[a, b], [c, b], [c, b]])] == ["b", "c", "a"]

rag_fusion = generar_versiones | recuperador.map() | rrf
for d, p in rag_fusion.invoke({"pregunta": pregunta})[:4]:
    print(f"{p:.4f} [{d.metadata['titulo']}] {d.page_content[:70]}...")

0.0481 [Agentes basados en LLM] Un agente autónomo usa un modelo de lenguaje como cerebro. El agente c...
0.0328 [Reflexión y autocrítica] La reflexión permite a un agente corregir sus errores a partir de la e...
0.0315 [Trazas y observabilidad] Una traza registra cada paso de una aplicación: las llamadas al modelo...
0.0312 [Trazas y observabilidad] Con las trazas se depuran fallos y se detectan los pasos más lentos o ...


## 9. Descomposición

Para preguntas compuestas, pedir al modelo que las divida en subpreguntas (una regla que corta por "y" delante de una partícula interrogativa) y responderlas de dos formas:
- **Recursiva**: cada subpregunta se responde con su contexto y con los pares pregunta-respuesta anteriores en el prompt.
- **Individual**: cada subpregunta se responde por separado y al final se unen las respuestas.

Mostrar el tamaño del prompt de cada paso en el modo recursivo.

In [10]:
def subpreguntas(m):
    partes = re.split(r"(?:,|\s+y)\s+(?=(?:qué|cómo|cuál|cuándo|quién|por qué)\b)", m.group(1).strip(" ¿?"))
    return "\n".join(f"¿{p[0].upper()}{p[1:]}?" for p in partes)


modelo = ModeloLocal(reglas=[(r"Divide en subpreguntas:\s*(.*)", subpreguntas)])
descomponer = ChatPromptTemplate.from_template("Divide en subpreguntas: {pregunta}") | modelo | StrOutputParser() | (lambda texto: texto.splitlines())

prompt_recursivo = ChatPromptTemplate.from_template(
    "Preguntas ya respondidas:\n{anteriores}\n\nContexto:\n{contexto}\n\nPregunta: {pregunta}"
)

compuesta = "¿Qué reduce RAG, cómo se indexan los documentos y qué mide la similitud del coseno?"
subs = descomponer.invoke({"pregunta": compuesta})
print(subs)
assert len(subs) == 3

anteriores = []
for sub in subs:
    valor = prompt_recursivo.invoke({"anteriores": "\n".join(anteriores) or "(ninguna)", "contexto": formatear(recuperador.invoke(sub)), "pregunta": sub})
    respuesta = modelo.invoke(valor).content
    print(f"[prompt de {len(valor.to_string())} caracteres] {sub} -> {respuesta}")
    anteriores.append(f"P: {sub}\nR: {respuesta}")

individuales = rag.batch(subs)
print("\nIndividual:\n" + "\n".join(f"- {s} {r}" for s, r in zip(subs, individuales)))
assert individuales == [a.split("\nR: ")[1] for a in anteriores]

['¿Qué reduce RAG?', '¿Cómo se indexan los documentos?', '¿Qué mide la similitud del coseno?']
[prompt de 646 caracteres] ¿Qué reduce RAG? -> RAG reduce las alucinaciones y permite usar información privada o reciente sin reentrenar el modelo.
[prompt de 810 caracteres] ¿Cómo se indexan los documentos? -> Para indexar, los documentos se dividen en fragmentos de unos cientos de caracteres con cierto solapamiento.
[prompt de 1087 caracteres] ¿Qué mide la similitud del coseno? -> La similitud del coseno mide el ángulo entre dos vectores y es la métrica más habitual.

Individual:
- ¿Qué reduce RAG? RAG reduce las alucinaciones y permite usar información privada o reciente sin reentrenar el modelo.
- ¿Cómo se indexan los documentos? Para indexar, los documentos se dividen en fragmentos de unos cientos de caracteres con cierto solapamiento.
- ¿Qué mide la similitud del coseno? La similitud del coseno mide el ángulo entre dos vectores y es la métrica más habitual.


## 10. Step-back

Las preguntas muy concretas pueden recuperar poco contexto. La técnica *step-back* pide al modelo una pregunta más general (con un prompt *few-shot*), recupera con las dos y responde con ambos contextos. Simular el paso atrás con una tabla de palabras clave y mostrar los títulos que aporta cada contexto.

In [11]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

GENERALES = [
    (r"HNSW|FAISS|vecinos", "¿Cómo guarda un agente su memoria a largo plazo?"),
    (r"ReAct|Reflexion", "¿Cómo corrige un agente sus errores?"),
    (r"MMR|coseno|ángulo", "¿Cómo funciona la recuperación por similitud?"),
]
modelo = ModeloLocal(reglas=[
    (r"(.+)", lambda m: next((g for p, g in GENERALES if re.search(p, m.group(1))), "¿Qué es RAG?")),
])
ejemplos = [
    {"pregunta": "¿El Teide es más alto que el Mulhacén?", "general": "¿Cuáles son las montañas más altas de España?"},
    {"pregunta": "¿Nació Pérez Galdós en Las Palmas?", "general": "¿Cuál es la biografía de Benito Pérez Galdós?"},
]
prompt_step_back = ChatPromptTemplate.from_messages([
    ("system", "Reformula la pregunta como una pregunta más general y fácil de responder."),
    FewShotChatMessagePromptTemplate(
        example_prompt=ChatPromptTemplate.from_messages([("human", "{pregunta}"), ("ai", "{general}")]), examples=ejemplos
    ),
    ("human", "{pregunta}"),
])
generalizar = prompt_step_back | modelo | StrOutputParser()

prompt_doble = ChatPromptTemplate.from_template("Contexto:\n{normal}\n\n{general}\n\nPregunta: {pregunta}")
step_back = (
    {"pregunta": RunnablePassthrough(), "general": RunnableLambda(lambda p: {"pregunta": p}) | generalizar}
    | RunnablePassthrough.assign(docs_normal=lambda x: recuperador.invoke(x["pregunta"]), docs_general=lambda x: recuperador.invoke(x["general"]))
    | RunnablePassthrough.assign(
        respuesta=RunnableLambda(lambda x: {"pregunta": x["pregunta"], "normal": formatear(x["docs_normal"]), "general": formatear(x["docs_general"])})
        | prompt_doble | ModeloLocal() | StrOutputParser()
    )
)
salida = step_back.invoke("¿Qué acelera HNSW?")
print("Pregunta general:", salida["general"])
print("Contexto normal: ", [d.metadata["titulo"] for d in salida["docs_normal"]])
print("Contexto general:", [d.metadata["titulo"] for d in salida["docs_general"]])
print("Respuesta:", salida["respuesta"])

Pregunta general: ¿Cómo guarda un agente su memoria a largo plazo?
Contexto normal:  ['Memoria de los agentes', 'Qué es RAG', 'Uso de herramientas', 'Trazas y observabilidad']
Contexto general: ['Memoria de los agentes', 'Agentes basados en LLM', 'Reflexión y autocrítica', 'Indexación y troceado']
Respuesta: Los algoritmos de vecinos más cercanos aproximados, como HNSW o FAISS, aceleran esa búsqueda en millones de vectores.


## 11. HyDE

Con HyDE (*Hypothetical Document Embeddings*) no se busca con el vector de la pregunta sino con el de un documento hipotético que la respondería, que se parece más a los documentos reales. Simular la escritura del documento con unas frases preparadas y comparar, para dos preguntas formuladas con palabras distintas a las del corpus, la posición del fragmento correcto buscando con la pregunta y con el documento hipotético.

In [12]:
CONOCIMIENTO = [
    (r"invent|falsa", "Esta técnica genera primero un documento hipotético que responde a la pregunta y después busca con el vector de ese documento."),
    (r"estructura|parecidos", "Los algoritmos de vecinos más cercanos aproximados, como HNSW, aceleran la búsqueda entre millones de vectores."),
]
modelo = ModeloLocal(reglas=[
    (r"Escribe un párrafo que responda a:\s*(.*)", lambda m: next((t for p, t in CONOCIMIENTO if re.search(p, m.group(1))), m.group(1))),
])
hyde = ChatPromptTemplate.from_template("Escribe un párrafo que responda a: {pregunta}") | modelo | StrOutputParser()


def posicion(consulta, condicion):
    docs = almacen.similarity_search(consulta, k=len(trozos))
    return next(i for i, d in enumerate(docs, 1) if condicion(d.page_content))


casos = [
    ("¿Qué método se inventa una respuesta falsa para buscar mejor?", lambda t: "HyDE" in t),
    ("¿Qué estructura encuentra rápido elementos parecidos?", lambda t: "HNSW" in t),
]
for pregunta, es_correcto in casos:
    hipotetico = hyde.invoke({"pregunta": pregunta})
    directa, con_hyde = posicion(pregunta, es_correcto), posicion(hipotetico, es_correcto)
    print(f"{pregunta}\n  documento hipotético: {hipotetico}\n  posición del fragmento correcto: {directa} con la pregunta, {con_hyde} con HyDE")
    assert con_hyde < directa

¿Qué método se inventa una respuesta falsa para buscar mejor?
  documento hipotético: Esta técnica genera primero un documento hipotético que responde a la pregunta y después busca con el vector de ese documento.
  posición del fragmento correcto: 4 con la pregunta, 1 con HyDE
¿Qué estructura encuentra rápido elementos parecidos?
  documento hipotético: Los algoritmos de vecinos más cercanos aproximados, como HNSW, aceleran la búsqueda entre millones de vectores.
  posición del fragmento correcto: 21 con la pregunta, 1 con HyDE
